# Prototype A (maquette): render combo frames 0050-0249 on Colab
Run top to bottom (the kit downloads from the public branch `render-kit-a`). Every cell is safe to re-run: finished frames are skipped.


In [ ]:
!nvidia-smi

In [ ]:
%%bash
# Blender 4.5.14 LTS (linux x64) + the X/GL libs it links against, once.
set -e
if [ ! -x /content/blender/blender ]; then
  apt-get -qq update && apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null
  wget -q https://download.blender.org/release/Blender4.5/blender-4.5.14-linux-x64.tar.xz -O /content/blender.tar.xz
  mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
  rm /content/blender.tar.xz
fi
/content/blender/blender --version | head -1

In [ ]:
import os, zipfile
if not os.path.isdir('/content/kit-a/scripts'):
    if not os.path.exists('/content/kit-a.zip'):
        os.system('wget -q https://github.com/mungvu2004/AppFront/raw/render-kit-a/kit-a.zip -O /content/kit-a.zip')
    zipfile.ZipFile('/content/kit-a.zip').extractall('/content/kit-a')
os.environ['KIT_ROOT'] = '/content/kit-a'
os.environ['KIT_OUT'] = '/content/out'
os.makedirs('/content/out', exist_ok=True)
print(sorted(os.listdir('/content/kit-a')))

In [ ]:
# GPU check: OptiX first, CUDA as fallback (the kit's render_settings does the same and prints '[render] device ...').
!/content/blender/blender -b --python-expr "import bpy;p=bpy.context.preferences.addons['cycles'].preferences;[print(k,[d.name for d in (p.get_devices_for_type(k) or [])]) for k in ('OPTIX','CUDA')]" 2>/dev/null | grep -E "OPTIX|CUDA" 

In [ ]:
import subprocess, time, re, os
BL = '/content/blender/blender'
def render(a, b, samples=160):
    cmd = [BL, '-b', '-P', '/content/kit-a/scripts/combo.py', '--', 'render', '--a', str(a), '--b', str(b), '--samples', str(samples)]
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ)
    for line in p.stdout:
        if line.startswith('[combo]') or line.startswith('[render]') or 'Error' in line or 'Traceback' in line:
            print(line.rstrip(), flush=True)
    p.wait()
    print(f'chunk {a}-{b - 1}: exit {p.returncode}, {time.time() - t0:.0f} s', flush=True)
# timing: 2 frames (also builds the scene once; first OptiX run compiles kernels)
render(50, 52)

In [ ]:
# all of 0050-0249 in chunks of 25 (skips frames already in /content/out)
for a in range(50, 250, 25):
    render(a, a + 25)
print('jpg frames:', len([f for f in os.listdir('/content/out') if f.endswith('.jpg')]))

In [ ]:
# WebP q80 at 1920 and 1280 -> /content/out/{1920,1280}/NNNN.webp
from PIL import Image
for w in (1920, 1280):
    os.makedirs(f'/content/out/{w}', exist_ok=True)
done = 0
for f in sorted(os.listdir('/content/out')):
    if not f.endswith('.jpg'):
        continue
    name = f[:-4] + '.webp'
    if os.path.exists(f'/content/out/1280/{name}'):
        continue
    im = Image.open(f'/content/out/{f}').convert('RGB')
    im.save(f'/content/out/1920/{name}', 'WEBP', quality=80, method=6)
    im.resize((1280, 720), Image.LANCZOS).save(f'/content/out/1280/{name}', 'WEBP', quality=80, method=6)
    done += 1
for w in (1920, 1280):
    fs = os.listdir(f'/content/out/{w}')
    print(w, len(fs), 'frames', round(sum(os.path.getsize(f'/content/out/{w}/{x}') for x in fs) / 1e6, 1), 'MB')

In [ ]:
# zip the WebP sets (paths 1920/NNNN.webp, 1280/NNNN.webp) and download
!cd /content/out && rm -f /content/a-frames.zip && zip -q -r /content/a-frames.zip 1920 1280 && ls -la /content/a-frames.zip
from google.colab import files
files.download('/content/a-frames.zip')